In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy pandas
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# 양자 커널 훈련

import TutorialFeedback from '@site/src/components/TutorialFeedback';



*사용 예상 시간: Eagle r3 프로세서에서 1분 미만 (참고: 이는 추정치이며, 실제 실행 시간은 다를 수 있습니다.)*
## 배경
이 튜토리얼에서는 이진 분류에 사용되는 양자 커널 행렬의 항목을 평가하기 위한 `Qiskit pattern`을 구축하는 방법을 설명합니다. `Qiskit patterns` 및 `Qiskit Serverless`를 사용하여 관리형 실행을 위해 클라우드에 배포하는 방법에 대한 자세한 내용은 [IBM Quantum&reg; Platform 문서 페이지](/guides/serverless)를 참조하세요.
## 요구 사항
이 튜토리얼을 시작하기 전에 다음이 설치되어 있는지 확인하세요:
- Qiskit SDK v1.0 이상 ([시각화](https://docs.quantum.ibm.com/api/qiskit/visualization) 지원 포함)
- Qiskit Runtime v0.22 이상 (`pip install qiskit-ibm-runtime`)
## 설정

In [ ]:
# General Imports and helper functions
import urllib.request

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.circuit.library import unitary_overlap
from qiskit.primitives import StatevectorSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, Sampler

# Download the dataset (portable across platforms)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/qiskit-community/prototype-quantum-kernel-training/main/data/dataset_graph7.csv",
    "dataset_graph7.csv",
)


def visualize_counts(res_counts, num_qubits, num_shots):
    """Visualize the outputs from the Qiskit Sampler primitive."""
    zero_prob = res_counts.get(0, 0.0)
    top_10 = dict(
        sorted(res_counts.items(), key=lambda item: item[1], reverse=True)[
            :10
        ]
    )
    top_10.update({0: zero_prob})
    by_key = dict(sorted(top_10.items(), key=lambda item: item[0]))
    x_vals, y_vals = list(zip(*by_key.items()))
    x_vals = [bin(x_val)[2:].zfill(num_qubits) for x_val in x_vals]
    y_vals_prob = []
    for t in range(len(y_vals)):
        y_vals_prob.append(y_vals[t] / num_shots)
    y_vals = y_vals_prob
    plt.bar(x_vals, y_vals)
    plt.xticks(rotation=75)
    plt.title("Results of sampling")
    plt.xlabel("Measured bitstring")
    plt.ylabel("Probability")
    plt.show()


def get_training_data():
    """Read the training data."""
    df = pd.read_csv("dataset_graph7.csv", sep=",", header=None)
    training_data = df.values[:20, :]
    ind = np.argsort(training_data[:, -1])
    X_train = training_data[ind][:, :-1]

    return X_train

## Small-scale simulator example

In this section, we walk through the four steps of the Qiskit pattern on a seven-qubit instance of the labeling-cosets-with-error problem and evaluate a single kernel matrix entry using the `StatevectorSampler` primitive from Qiskit. A statevector simulator is exact (up to shot noise) and shows us the method end-to-end without consuming QPU time. We then repeat the same instance on real hardware in the hardware example section.

### Step 1: Map classical inputs to a quantum problem

*   Input: Training dataset.
*   Output: Abstract circuit for calculating a kernel matrix entry.

The binary classification problem we aim to solve here is referred to as "[_labeling cosets with error_](https://arxiv.org/abs/2105.03406)." The input training dataset contains a group structure, consisting of two cosets formed by a group and subgroup.
The group is taken to be $G = SU(2)^{\otimes n}$ for qubits, which is the special unitary group of
$2 \times 2$ matrices and has wide applicability in nature; e.g., the Standard Model of particle physics.
We take the (graph-stabilizer) subgroup $S_\text{graph} < G$ with $S_\text{graph} = \langle \{ X_i \otimes _{k:(k,i) \in \mathcal{E}} Z_k\} _{i \in \mathcal{V}} \} \rangle$ for a graph with edges $\mathcal{E}$ and vertices $\mathcal{V}$.
Note that the stabilizers fix a stabilizer state such that $D_s | \psi \rangle = | \psi \rangle,~ \forall s \in S_\text{graph}$.
Finally, we define two left-cosets $C_\pm = c_\pm S_\text{graph}$ by drawing two $c_\pm \in G$ at random.

For more details about the dataset and how it is generated, see [this notebook](https://github.com/qiskit-community/prototype-quantum-kernel-training/blob/main/docs/background/qkernels_and_data_w_group_structure.ipynb) from the [Quantum Kernel Training Toolkit](https://github.com/qiskit-community/prototype-quantum-kernel-training/tree/main).

We create the quantum circuit used to evaluate one entry in the kernel matrix.
The input data is used to determine the rotation angles for the circuit's parametrized gates.
For simplicity, we will use data samples `x1=14` and `x2=19`.

***Note: The dataset used in this tutorial can be downloaded [here](https://github.com/qiskit-community/prototype-quantum-kernel-training/blob/main/data/dataset_graph7.csv).***

In [2]:
# Prepare training data
X_train = get_training_data()

# Empty kernel matrix
num_samples = np.shape(X_train)[0]
kernel_matrix = np.full((num_samples, num_samples), np.nan)

# Prepare feature map for computing overlap
num_features = np.shape(X_train)[1]
num_qubits = int(num_features / 2)
entangler_map = [[0, 2], [3, 4], [2, 5], [1, 4], [2, 3], [4, 6]]
fm = QuantumCircuit(num_qubits)
training_param = Parameter("θ")
feature_params = ParameterVector("x", num_qubits * 2)
fm.ry(training_param, fm.qubits)
for cz in entangler_map:
    fm.cz(cz[0], cz[1])
for i in range(num_qubits):
    fm.rz(-2 * feature_params[2 * i + 1], i)
    fm.rx(-2 * feature_params[2 * i], i)

# Assign tunable parameter to known optimal value and set the data params for first two samples
x1 = 14
x2 = 19
unitary1 = fm.assign_parameters(list(X_train[x1]) + [np.pi / 2])
unitary2 = fm.assign_parameters(list(X_train[x2]) + [np.pi / 2])

# Create the overlap circuit
overlap_circ = unitary_overlap(unitary1, unitary2)
overlap_circ.measure_all()
overlap_circ.draw("mpl", scale=0.6, style="iqp")

<Image src="../docs/images/tutorials/quantum-kernel-training/extracted-outputs/70d6faff-9a56-44bb-b26f-f573a8c90889-0.avif" alt="Output of the previous code cell" />

## 1단계: 고전적 입력을 양자 문제로 변환
*   입력: 훈련 데이터셋
*   출력: 커널 행렬 항목 계산을 위한 추상 Circuit

커널 행렬의 한 항목을 평가하는 데 사용되는 양자 Circuit을 생성합니다. 입력 데이터를 사용하여 Circuit의 매개변수화된 게이트에 대한 회전 각도를 결정합니다. 데이터 샘플 `x1=14`와 `x2=19`를 사용하겠습니다.

***참고: 이 튜토리얼에 사용된 데이터셋은 [여기](https://github.com/qiskit-community/prototype-quantum-kernel-training/blob/main/data/dataset_graph7.csv)에서 다운로드할 수 있습니다.***

In [3]:
sampler = StatevectorSampler()

# Execute and get counts
num_shots = 10_000
results = sampler.run([overlap_circ], shots=num_shots).result()
counts = results[0].data.meas.get_int_counts()

# Plot counts
visualize_counts(counts, num_qubits, num_shots)

<Image src="../docs/images/tutorials/quantum-kernel-training/extracted-outputs/step3-small-code-0.avif" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/tutorials/quantum-kernel-training/extracted-outputs/70d6faff-9a56-44bb-b26f-f573a8c90889-0.avif)

## 2단계: 양자 하드웨어 실행을 위한 문제 최적화
*   입력: 특정 Backend에 최적화되지 않은 추상 Circuit
*   출력: 선택된 QPU에 최적화된 목표 Circuit과 관측량

Qiskit의 `generate_preset_pass_manager` 함수를 사용하여 실험을 실행할 QPU에 맞는 Circuit 최적화 루틴을 지정합니다. 가장 높은 수준의 최적화를 제공하는 사전 설정 패스 매니저를 사용하도록 `optimization_level=3`으로 설정합니다.

In [4]:
kernel_matrix[x1, x2] = counts.get(0, 0.0) / num_shots
print(f"Fidelity (simulator): {kernel_matrix[x1, x2]}")

Fidelity (simulator): 0.8261


![이전 코드 셀의 출력](../docs/images/tutorials/quantum-kernel-training/extracted-outputs/49607b34-9723-493d-85da-bd97c1351104-0.avif)

## 3단계: Qiskit 프리미티브를 사용한 실행
*   입력: 목표 Circuit
*   출력: 준확률 분포

Qiskit Runtime의 `Sampler` 프리미티브를 사용하여 Circuit 샘플링으로부터 얻은 상태의 준확률 분포를 재구성합니다. 커널 행렬 생성 작업에서는 특히 |0> 상태를 측정할 확률에 관심이 있습니다.

이 데모에서는 `qiskit-ibm-runtime` 프리미티브를 사용하여 QPU에서 실행합니다. `qiskit` statevector 기반 프리미티브로 실행하려면, Qiskit IBM&reg; Runtime 프리미티브를 사용하는 코드 블록을 주석 처리된 블록으로 교체하세요.

In [5]:
# ------------------------------ Step 1 ------------------------------
# Prepare training data
X_train = get_training_data()

# Empty kernel matrix
num_samples = np.shape(X_train)[0]
kernel_matrix = np.full((num_samples, num_samples), np.nan)

# Prepare feature map for computing overlap
num_features = np.shape(X_train)[1]
num_qubits = int(num_features / 2)
entangler_map = [[0, 2], [3, 4], [2, 5], [1, 4], [2, 3], [4, 6]]
fm = QuantumCircuit(num_qubits)
training_param = Parameter("θ")
feature_params = ParameterVector("x", num_qubits * 2)
fm.ry(training_param, fm.qubits)
for cz in entangler_map:
    fm.cz(cz[0], cz[1])
for i in range(num_qubits):
    fm.rz(-2 * feature_params[2 * i + 1], i)
    fm.rx(-2 * feature_params[2 * i], i)

# Assign tunable parameter to known optimal value and set the data params for first two samples
x1 = 14
x2 = 19
unitary1 = fm.assign_parameters(list(X_train[x1]) + [np.pi / 2])
unitary2 = fm.assign_parameters(list(X_train[x2]) + [np.pi / 2])

# Create the overlap circuit
overlap_circ = unitary_overlap(unitary1, unitary2)
overlap_circ.measure_all()

# ------------------------------ Step 2 ------------------------------
service = QiskitRuntimeService()
# backend = service.least_busy(
#    operational=True, simulator=False, min_num_qubits=overlap_circ.num_qubits
# )
backend = service.backend("ibm_pittsburgh")
print(f"Using backend: {backend.name}")
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
overlap_ibm = pm.run(overlap_circ)

# ------------------------------ Step 3 ------------------------------
sampler = Sampler(mode=backend)
sampler.options.environment.job_tags = ["TUT_QKT"]

num_shots = 10_000
results = sampler.run([overlap_ibm], shots=num_shots).result()
counts = results[0].data.meas.get_int_counts()
visualize_counts(counts, num_qubits, num_shots)

# ------------------------------ Step 4 ------------------------------
kernel_matrix[x1, x2] = counts.get(0, 0.0) / num_shots
print(f"Fidelity (hardware): {kernel_matrix[x1, x2]}")

Using backend: ibm_pittsburgh


<Image src="../docs/images/tutorials/quantum-kernel-training/extracted-outputs/d2f4f6cf-067e-4d53-aa04-7ca9c803d3e1-1.avif" alt="Output of the previous code cell" />

Fidelity (hardware): 0.7517


![이전 코드 셀의 출력](../docs/images/tutorials/quantum-kernel-training/extracted-outputs/d2f4f6cf-067e-4d53-aa04-7ca9c803d3e1-0.avif)

## 4단계: 후처리 및 원하는 고전적 형식으로 결과 반환

*   입력: 확률 분포
*   출력: 단일 커널 행렬 요소

오버랩 Circuit에서 |0>을 측정할 확률을 계산하고, 이 특정 오버랩 Circuit이 나타내는 샘플에 해당하는 위치(행 15, 열 20)에 커널 행렬을 채웁니다. 이 시각화에서 더 진한 빨간색은 1.0에 가까운 충실도를 나타냅니다. 전체 커널 행렬을 채우려면 각 항목에 대해 양자 실험을 실행해야 합니다.